# Notebook 05: Proposed EfficientNetV2-S Training — Main19

This notebook trains the primary 19-class crop-disease classifier.

## Protocol

- Training images: `main19_train.csv` only
- Model selection, early stopping, and threshold-free comparison:
  `main19_calibration.csv` only
- Internal-test manifest: deliberately not loaded in this notebook
- Primary checkpoint selection metric: calibration macro-F1
- Training loss: class-balanced focal cross-entropy

The internal test set will be used once later, after the final proposed configuration
has been selected.

In [ ]:
import os
import gc
import json
import random
import platform
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 7

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("PyTorch:", torch.__version__)
print("Torchvision:", __import__("torchvision").__version__)

if DEVICE.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_capability = torch.cuda.get_device_capability(0)

    print("GPU:", gpu_name)
    print("CUDA capability:", gpu_capability)
    print("GPU memory (GB):", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3,
        2
    ))

    assert gpu_capability >= (7, 0), (
        f"{gpu_name} has CUDA capability {gpu_capability}. "
        "Use a T4 or newer GPU for this PyTorch build."
    )
else:
    raise RuntimeError(
        "GPU not detected. Enable a Kaggle T4 GPU before training."
    )

In [ ]:
# ---------------------------------------------------
# Global professional paper-style plotting configuration
# ---------------------------------------------------
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,

    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,

    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,

    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",

    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})


def paper_axes(ax):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.7)

    for spine in ax.spines.values():
        spine.set_linewidth(1.2)

    ax.tick_params(which="both", direction="in", top=True, right=True)


np.random.seed(SEED)

print("Professional paper plotting style enabled.")

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
WORK_DIR = Path("/kaggle/working")

OUTPUT_DIR = WORK_DIR / "proposed_efficientnetv2s_outputs"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
METADATA_DIR = OUTPUT_DIR / "metadata"

for folder in [
    OUTPUT_DIR,
    CHECKPOINT_DIR,
    FIG_DIR,
    TABLE_DIR,
    METADATA_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Input root:", INPUT_ROOT)
print("Output directory:", OUTPUT_DIR)

In [ ]:
def find_input_file(file_name):
    matches = list(INPUT_ROOT.rglob(file_name))

    if len(matches) == 0:
        raise FileNotFoundError(
            f"Could not locate '{file_name}' under /kaggle/input.\n"
            "Attach the Notebook-02 split_outputs dataset."
        )

    if len(matches) > 1:
        print(f"Multiple matches found for {file_name}:")
        for match in matches:
            print(" -", match)
        print("\nUsing first match:", matches[0])

    return matches[0]


TRAIN_MANIFEST_PATH = find_input_file("main19_train.csv")
CALIBRATION_MANIFEST_PATH = find_input_file(
    "main19_calibration.csv"
)
CLASS_MAPPING_PATH = find_input_file(
    "class_to_index_main19.json"
)
PROTOCOL_PATH = find_input_file("data_protocol.json")

print("Train manifest:", TRAIN_MANIFEST_PATH)
print("Calibration manifest:", CALIBRATION_MANIFEST_PATH)
print("Class mapping:", CLASS_MAPPING_PATH)
print("Data protocol:", PROTOCOL_PATH)

In [ ]:
train_df = pd.read_csv(TRAIN_MANIFEST_PATH)
calibration_df = pd.read_csv(CALIBRATION_MANIFEST_PATH)

with open(CLASS_MAPPING_PATH, "r") as file:
    class_to_index = json.load(file)

with open(PROTOCOL_PATH, "r") as file:
    data_protocol = json.load(file)

class_to_index = {
    str(label): int(index)
    for label, index in class_to_index.items()
}

index_to_class = {
    int(index): label
    for label, index in class_to_index.items()
}

NUM_CLASSES = len(class_to_index)

required_columns = {
    "image_path",
    "label",
    "final_split"
}

for dataframe_name, dataframe in [
    ("training", train_df),
    ("calibration", calibration_df)
]:
    missing_columns = required_columns - set(dataframe.columns)

    assert not missing_columns, (
        f"{dataframe_name} manifest is missing: {missing_columns}"
    )

assert set(train_df["final_split"]) == {"train"}
assert set(calibration_df["final_split"]) == {"calibration"}

assert train_df["label"].nunique() == 19
assert calibration_df["label"].nunique() == 19
assert NUM_CLASSES == 19

assert set(train_df["label"]) == set(class_to_index)
assert set(calibration_df["label"]) == set(class_to_index)

assert train_df["image_path"].nunique() == len(train_df)
assert calibration_df["image_path"].nunique() == len(calibration_df)

assert set(train_df["image_path"]).isdisjoint(
    set(calibration_df["image_path"])
), "Training and calibration image paths overlap."

assert set(class_to_index.values()) == set(range(NUM_CLASSES))

print("Training images:", len(train_df))
print("Calibration images:", len(calibration_df))
print("Classes:", NUM_CLASSES)

print("\nPASS: Data partitions are valid and disjoint.")
print("Internal-test manifest is intentionally not loaded.")

In [ ]:
train_exists = train_df["image_path"].map(
    lambda path: Path(path).exists()
)

calibration_exists = calibration_df["image_path"].map(
    lambda path: Path(path).exists()
)

print("Existing training images:", int(train_exists.sum()), "/", len(train_df))
print(
    "Existing calibration images:",
    int(calibration_exists.sum()),
    "/",
    len(calibration_df)
)

assert train_exists.all(), (
    f"{int((~train_exists).sum())} training image paths do not exist."
)

assert calibration_exists.all(), (
    f"{int((~calibration_exists).sum())} calibration image paths do not exist."
)

print("\nPASS: All source images are accessible.")

In [ ]:
input_record = {
    "notebook": "05_proposed_efficientnet_training",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "train_manifest": str(TRAIN_MANIFEST_PATH),
    "calibration_manifest": str(CALIBRATION_MANIFEST_PATH),
    "class_mapping": str(CLASS_MAPPING_PATH),
    "data_protocol": str(PROTOCOL_PATH),
    "training_images": int(len(train_df)),
    "calibration_images": int(len(calibration_df)),
    "classes": int(NUM_CLASSES),
    "internal_test_loaded": False,
    "confirmed_near_duplicate_exclusions": data_protocol.get(
        "confirmed_near_duplicate_exclusions",
        "not_recorded"
    )
}

input_record_path = METADATA_DIR / "input_data_record.json"

with open(input_record_path, "w") as file:
    json.dump(input_record, file, indent=2)

print(json.dumps(input_record, indent=2))

In [ ]:
ordered_labels = sorted(
    class_to_index,
    key=class_to_index.get
)

class_counts = (
    train_df["label"]
    .value_counts()
    .reindex(ordered_labels)
)

class_weight_values = len(train_df) / (
    NUM_CLASSES * class_counts.values.astype(np.float32)
)

class_weights = torch.tensor(
    class_weight_values,
    dtype=torch.float32,
    device=DEVICE
)

class_weight_df = pd.DataFrame({
    "class_index": [class_to_index[label] for label in ordered_labels],
    "label": ordered_labels,
    "train_images": class_counts.values,
    "cross_entropy_weight": class_weight_values
})

class_weight_path = TABLE_DIR / "proposed_model_class_weights.csv"
class_weight_df.to_csv(class_weight_path, index=False)

display(class_weight_df)

print("Minimum class count:", int(class_counts.min()))
print("Maximum class count:", int(class_counts.max()))
print(
    "Imbalance ratio:",
    round(float(class_counts.max() / class_counts.min()), 2)
)
print("Saved:", class_weight_path)

In [ ]:
plot_df = class_weight_df.sort_values(
    "train_images",
    ascending=True
)

fig, ax = plt.subplots(figsize=(13, 6))

ax.barh(
    plot_df["label"],
    plot_df["train_images"],
    color="#2F6B9A",
    edgecolor="black",
    linewidth=0.65,
    zorder=3
)

ax.set_xlabel("Number of training images")
ax.set_ylabel("Main19 crop-disease class")
ax.set_title(
    "Training-set class distribution for the proposed Main19 classifier",
    pad=10
)

paper_axes(ax)

for index, value in enumerate(plot_df["train_images"]):
    ax.text(
        value + max(plot_df["train_images"]) * 0.008,
        index,
        f"{int(value)}",
        va="center",
        ha="left",
        fontsize=9
    )

ax.set_xlim(0, max(plot_df["train_images"]) * 1.13)

plt.tight_layout()

figure_path = FIG_DIR / "fig_01_training_class_distribution.png"
pdf_path = FIG_DIR / "fig_01_training_class_distribution.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)
print("Saved:", pdf_path)

In [ ]:
IMAGE_SIZE = 384

TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((432, 432)),
    transforms.RandomResizedCrop(
        IMAGE_SIZE,
        scale=(0.78, 1.00),
        ratio=(0.90, 1.10)
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(degrees=18),
    transforms.ColorJitter(
        brightness=0.18,
        contrast=0.18,
        saturation=0.12,
        hue=0.03
    ),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.04, 0.04),
        scale=(0.95, 1.05),
        shear=5
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    ),
    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.08),
        ratio=(0.30, 3.30),
        value="random"
    )
])

EVAL_TRANSFORM = transforms.Compose([
    transforms.Resize((432, 432)),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Model input size:", IMAGE_SIZE, "x", IMAGE_SIZE)

In [ ]:
class PlantDiseaseDataset(Dataset):
    def __init__(self, dataframe, class_to_index, transform):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.class_to_index = class_to_index
        self.transform = transform

        self.image_paths = self.dataframe["image_path"].tolist()
        self.labels = self.dataframe["label"].tolist()
        self.targets = [
            self.class_to_index[label]
            for label in self.labels
        ]

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        image_path = self.image_paths[index]
        target = self.targets[index]

        try:
            with Image.open(image_path) as image:
                image = image.convert("RGB")
        except Exception as error:
            raise RuntimeError(
                f"Failed to load index {index}: {image_path}"
            ) from error

        image = self.transform(image)

        return {
            "image": image,
            "target": torch.tensor(target, dtype=torch.long),
            "path": image_path
        }


train_dataset = PlantDiseaseDataset(
    dataframe=train_df,
    class_to_index=class_to_index,
    transform=TRAIN_TRANSFORM
)

calibration_dataset = PlantDiseaseDataset(
    dataframe=calibration_df,
    class_to_index=class_to_index,
    transform=EVAL_TRANSFORM
)

print("Training dataset:", len(train_dataset))
print("Calibration dataset:", len(calibration_dataset))

In [ ]:
inverse_normalize = transforms.Normalize(
    mean=[
        -0.485 / 0.229,
        -0.456 / 0.224,
        -0.406 / 0.225
    ],
    std=[
        1 / 0.229,
        1 / 0.224,
        1 / 0.225
    ]
)

sample_indices = np.random.choice(
    len(train_dataset),
    size=6,
    replace=False
)

fig, axes = plt.subplots(2, 3, figsize=(13, 8))

for axis, sample_index in zip(axes.flat, sample_indices):
    sample = train_dataset[sample_index]

    image = inverse_normalize(
        sample["image"]
    ).clamp(0, 1)

    class_name = index_to_class[
        int(sample["target"])
    ]

    axis.imshow(image.permute(1, 2, 0))
    axis.set_title(
        class_name.replace("__", " — ").replace("_", " "),
        fontsize=10,
        pad=7
    )
    axis.set_xticks([])
    axis.set_yticks([])

    for spine in axis.spines.values():
        spine.set_linewidth(1.0)

fig.suptitle(
    "Examples of training-time image augmentation",
    y=0.99,
    fontsize=14
)

plt.tight_layout()

figure_path = FIG_DIR / "fig_02_training_augmentation_examples.png"
pdf_path = FIG_DIR / "fig_02_training_augmentation_examples.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)

In [ ]:
BATCH_SIZE = 12
NUM_WORKERS = min(4, os.cpu_count() or 2)
PIN_MEMORY = DEVICE.type == "cuda"

loader_generator = torch.Generator()
loader_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0,
    generator=loader_generator
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    persistent_workers=NUM_WORKERS > 0
)

print("Batch size:", BATCH_SIZE)
print("DataLoader workers:", NUM_WORKERS)
print("Training batches:", len(train_loader))
print("Calibration batches:", len(calibration_loader))

In [ ]:
class FocalCrossEntropyLoss(nn.Module):
    def __init__(self, class_weights=None, gamma=1.5):
        super().__init__()
        self.gamma = gamma
        self.register_buffer(
            "class_weights",
            class_weights if class_weights is not None else None
        )

    def forward(self, logits, targets):
        log_probabilities = F.log_softmax(logits, dim=1)

        negative_log_likelihood = F.nll_loss(
            log_probabilities,
            targets,
            weight=self.class_weights,
            reduction="none"
        )

        probabilities = torch.exp(log_probabilities)
        probability_true_class = probabilities.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        focal_weight = (
            1.0 - probability_true_class
        ).pow(self.gamma)

        return (focal_weight * negative_log_likelihood).mean()


print("FocalCrossEntropyLoss defined.")

In [ ]:
try:
    pretrained_weights = (
        models.EfficientNet_V2_S_Weights.IMAGENET1K_V1
    )

    model = models.efficientnet_v2_s(
        weights=pretrained_weights
    )

    print("Loaded ImageNet-pretrained EfficientNetV2-S.")

except Exception as error:
    raise RuntimeError(
        "Could not load EfficientNetV2-S pretrained weights. "
        "Enable Internet or attach the required PyTorch model weights."
    ) from error

classifier_input_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(p=0.35),
    nn.Linear(classifier_input_features, NUM_CLASSES)
)

model = model.to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Total parameters:", f"{total_parameters:,}")
print("Trainable parameters:", f"{trainable_parameters:,}")
print("\nClassifier:")
print(model.classifier)

In [ ]:
EPOCHS = 20
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 5
FOCAL_GAMMA = 1.5
MAX_GRAD_NORM = 1.0

criterion = FocalCrossEntropyLoss(
    class_weights=class_weights,
    gamma=FOCAL_GAMMA
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2,
    min_lr=1e-6
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=DEVICE.type == "cuda"
)

training_config = {
    "architecture": "EfficientNetV2-S",
    "pretrained_weights": "IMAGENET1K_V1",
    "num_classes": NUM_CLASSES,
    "input_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs_maximum": EPOCHS,
    "optimizer": "AdamW",
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "loss": "class-weighted focal cross-entropy",
    "focal_gamma": FOCAL_GAMMA,
    "early_stopping_patience": EARLY_STOPPING_PATIENCE,
    "checkpoint_selection_metric": "calibration macro-F1",
    "seed": SEED,
    "internal_test_used": False,
    "device": str(DEVICE)
}

training_config_path = METADATA_DIR / "training_config.json"

with open(training_config_path, "w") as file:
    json.dump(training_config, file, indent=2)

print(json.dumps(training_config, indent=2))

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):
    model.train()

    running_loss = 0.0
    all_targets = []
    all_predictions = []

    progress_bar = tqdm(
        loader,
        desc="Training",
        leave=False
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=device.type,
            enabled=device.type == "cuda"
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=MAX_GRAD_NORM
        )

        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)

        predictions = logits.argmax(dim=1)

        all_targets.extend(
            targets.detach().cpu().numpy()
        )

        all_predictions.extend(
            predictions.detach().cpu().numpy()
        )

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            all_targets,
            all_predictions
        ),
        "macro_f1": f1_score(
            all_targets,
            all_predictions,
            average="macro",
            zero_division=0
        )
    }


@torch.no_grad()
def evaluate_model(
    model,
    loader,
    criterion,
    device
):
    model.eval()

    running_loss = 0.0
    all_targets = []
    all_predictions = []
    all_probabilities = []
    all_paths = []

    progress_bar = tqdm(
        loader,
        desc="Calibration",
        leave=False
    )

    for batch in progress_bar:
        images = batch["image"].to(
            device,
            non_blocking=True
        )

        targets = batch["target"].to(
            device,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=device.type,
            enabled=device.type == "cuda"
        ):
            logits = model(images)
            loss = criterion(logits, targets)

        probabilities = torch.softmax(
            logits,
            dim=1
        )

        predictions = probabilities.argmax(dim=1)

        running_loss += loss.item() * images.size(0)

        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

        all_paths.extend(batch["path"])

    all_probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )

    metrics = {
        "loss": running_loss / len(loader.dataset),
        "accuracy": accuracy_score(
            all_targets,
            all_predictions
        ),
        "balanced_accuracy": balanced_accuracy_score(
            all_targets,
            all_predictions
        ),
        "macro_f1": f1_score(
            all_targets,
            all_predictions,
            average="macro",
            zero_division=0
        )
    }

    outputs = {
        "targets": np.array(all_targets),
        "predictions": np.array(all_predictions),
        "probabilities": all_probabilities,
        "paths": all_paths
    }

    return metrics, outputs

In [ ]:
best_macro_f1 = -np.inf
best_epoch = 0
epochs_without_improvement = 0
history = []

best_checkpoint_path = (
    CHECKPOINT_DIR / "best_efficientnetv2s_main19.pt"
)

for epoch in range(1, EPOCHS + 1):
    print(f"\n{'=' * 78}")
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"{'=' * 78}")

    train_metrics = train_one_epoch(
        model=model,
        loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        scaler=scaler,
        device=DEVICE
    )

    calibration_metrics, _ = evaluate_model(
        model=model,
        loader=calibration_loader,
        criterion=criterion,
        device=DEVICE
    )

    current_lr = optimizer.param_groups[0]["lr"]

    scheduler.step(calibration_metrics["macro_f1"])

    epoch_record = {
        "epoch": epoch,
        "learning_rate": current_lr,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_macro_f1": train_metrics["macro_f1"],
        "calibration_loss": calibration_metrics["loss"],
        "calibration_accuracy": calibration_metrics["accuracy"],
        "calibration_balanced_accuracy": calibration_metrics[
            "balanced_accuracy"
        ],
        "calibration_macro_f1": calibration_metrics[
            "macro_f1"
        ]
    }

    history.append(epoch_record)

    print(
        f"Train | loss={train_metrics['loss']:.5f}, "
        f"accuracy={train_metrics['accuracy']:.5f}, "
        f"macro-F1={train_metrics['macro_f1']:.5f}"
    )

    print(
        f"Calibration | loss={calibration_metrics['loss']:.5f}, "
        f"accuracy={calibration_metrics['accuracy']:.5f}, "
        f"balanced accuracy="
        f"{calibration_metrics['balanced_accuracy']:.5f}, "
        f"macro-F1={calibration_metrics['macro_f1']:.5f}"
    )

    if calibration_metrics["macro_f1"] > best_macro_f1:
        best_macro_f1 = calibration_metrics["macro_f1"]
        best_epoch = epoch
        epochs_without_improvement = 0

        checkpoint = {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_calibration_macro_f1": best_macro_f1,
            "class_to_index": class_to_index,
            "training_config": training_config,
            "input_record": input_record
        }

        torch.save(
            checkpoint,
            best_checkpoint_path
        )

        print(
            "Saved best checkpoint: "
            f"calibration macro-F1={best_macro_f1:.5f}"
        )

    else:
        epochs_without_improvement += 1

        print(
            f"No improvement: {epochs_without_improvement}/"
            f"{EARLY_STOPPING_PATIENCE}"
        )

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print("\nEarly stopping triggered.")
        break

    gc.collect()
    torch.cuda.empty_cache()

history_df = pd.DataFrame(history)

history_path = TABLE_DIR / "proposed_training_history.csv"
history_df.to_csv(history_path, index=False)

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best calibration macro-F1:", round(best_macro_f1, 6))
print("Checkpoint:", best_checkpoint_path)
print("History:", history_path)

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(13, 6)
)

axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    marker="o",
    markersize=5,
    linewidth=2.0,
    color="#1F77B4",
    label="Training"
)

axes[0].plot(
    history_df["epoch"],
    history_df["calibration_loss"],
    marker="s",
    markersize=5,
    linewidth=2.0,
    color="#D55E00",
    label="Calibration"
)

axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Focal cross-entropy loss")
axes[0].set_title("Optimisation trajectory", pad=10)
axes[0].legend(
    loc="best",
    fancybox=False,
    borderpad=0.8
)

paper_axes(axes[0])

axes[1].plot(
    history_df["epoch"],
    history_df["train_macro_f1"],
    marker="o",
    markersize=5,
    linewidth=2.0,
    color="#1F77B4",
    label="Training macro-F1"
)

axes[1].plot(
    history_df["epoch"],
    history_df["calibration_macro_f1"],
    marker="s",
    markersize=5,
    linewidth=2.0,
    color="#D55E00",
    label="Calibration macro-F1"
)

axes[1].axvline(
    best_epoch,
    color="black",
    linewidth=1.2,
    linestyle="--",
    label=f"Selected epoch ({best_epoch})"
)

axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro-F1")
axes[1].set_ylim(
    max(0.0, history_df[
        ["train_macro_f1", "calibration_macro_f1"]
    ].min().min() - 0.02),
    1.005
)

axes[1].set_title(
    "Classification performance during training",
    pad=10
)

axes[1].legend(
    loc="lower right",
    fancybox=False,
    borderpad=0.8
)

paper_axes(axes[1])

fig.suptitle(
    "EfficientNetV2-S training and calibration performance",
    y=1.02,
    fontsize=14
)

plt.tight_layout()

figure_path = FIG_DIR / "fig_03_proposed_learning_curves.png"
pdf_path = FIG_DIR / "fig_03_proposed_learning_curves.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)
print("Saved:", pdf_path)

In [ ]:
best_checkpoint = torch.load(
    best_checkpoint_path,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    best_checkpoint["model_state_dict"]
)

model.eval()

print("Checkpoint reloaded.")
print("Selected epoch:", best_checkpoint["epoch"])
print(
    "Selected calibration macro-F1:",
    round(
        best_checkpoint["best_calibration_macro_f1"],
        6
    )
)

In [ ]:
final_calibration_metrics, final_calibration_outputs = evaluate_model(
    model=model,
    loader=calibration_loader,
    criterion=criterion,
    device=DEVICE
)

final_metrics_df = pd.DataFrame([{
    "model": "EfficientNetV2-S",
    "split": "calibration",
    "selected_epoch": best_epoch,
    "loss": final_calibration_metrics["loss"],
    "accuracy": final_calibration_metrics["accuracy"],
    "balanced_accuracy": final_calibration_metrics[
        "balanced_accuracy"
    ],
    "macro_f1": final_calibration_metrics["macro_f1"]
}])

final_metrics_path = (
    TABLE_DIR / "proposed_model_calibration_metrics.csv"
)

final_metrics_df.to_csv(
    final_metrics_path,
    index=False
)

display(final_metrics_df)

print("Saved:", final_metrics_path)

In [ ]:
class_names = [
    index_to_class[index]
    for index in range(NUM_CLASSES)
]

calibration_report = classification_report(
    final_calibration_outputs["targets"],
    final_calibration_outputs["predictions"],
    labels=list(range(NUM_CLASSES)),
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

calibration_report_df = (
    pd.DataFrame(calibration_report)
    .transpose()
    .reset_index()
    .rename(columns={"index": "class_or_average"})
)

report_path = TABLE_DIR / "proposed_calibration_report.csv"
calibration_report_df.to_csv(
    report_path,
    index=False
)

prediction_df = calibration_df[
    ["image_path", "label"]
].copy()

prediction_df["true_class_index"] = (
    final_calibration_outputs["targets"]
)

prediction_df["predicted_class_index"] = (
    final_calibration_outputs["predictions"]
)

prediction_df["predicted_label"] = [
    index_to_class[index]
    for index in final_calibration_outputs["predictions"]
]

prediction_df["confidence"] = (
    final_calibration_outputs["probabilities"].max(axis=1)
)

for class_index in range(NUM_CLASSES):
    class_name = index_to_class[class_index]

    safe_name = (
        class_name
        .replace("__", "_")
        .replace(" ", "_")
    )

    prediction_df[
        f"probability_{class_index:02d}_{safe_name}"
    ] = final_calibration_outputs[
        "probabilities"
    ][:, class_index]

prediction_path = (
    TABLE_DIR / "proposed_calibration_predictions.csv"
)

prediction_df.to_csv(
    prediction_path,
    index=False
)

display(calibration_report_df.head())
display(prediction_df.head())

print("Saved:", report_path)
print("Saved:", prediction_path)

In [ ]:
confusion = confusion_matrix(
    final_calibration_outputs["targets"],
    final_calibration_outputs["predictions"],
    labels=list(range(NUM_CLASSES)),
    normalize="true"
)

readable_class_names = [
    class_name
    .replace("__", " — ")
    .replace("_", " ")
    for class_name in class_names
]

fig, ax = plt.subplots(figsize=(13, 11))

image = ax.imshow(
    confusion,
    cmap="Blues",
    interpolation="nearest",
    vmin=0.0,
    vmax=1.0,
    aspect="auto"
)

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04
)

colorbar.set_label(
    "Row-normalised proportion",
    rotation=90,
    labelpad=12
)

ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_yticks(np.arange(NUM_CLASSES))

ax.set_xticklabels(
    readable_class_names,
    rotation=90,
    ha="center",
    fontsize=8
)

ax.set_yticklabels(
    readable_class_names,
    fontsize=8
)

ax.set_xlabel("Predicted class")
ax.set_ylabel("True class")

ax.set_title(
    "Calibration confusion matrix for selected EfficientNetV2-S",
    pad=10
)

for spine in ax.spines.values():
    spine.set_linewidth(1.2)

ax.tick_params(
    which="both",
    direction="in",
    top=True,
    right=True
)

plt.tight_layout()

figure_path = FIG_DIR / "fig_04_proposed_calibration_confusion_matrix.png"
pdf_path = FIG_DIR / "fig_04_proposed_calibration_confusion_matrix.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)
print("Saved:", pdf_path)

In [ ]:
per_class_f1 = calibration_report_df[
    calibration_report_df["class_or_average"].isin(class_names)
].copy()

per_class_f1["readable_label"] = (
    per_class_f1["class_or_average"]
    .str.replace("__", " — ", regex=False)
    .str.replace("_", " ", regex=False)
)

per_class_f1 = per_class_f1.sort_values(
    "f1-score",
    ascending=True
)

fig, ax = plt.subplots(figsize=(13, 6))

ax.barh(
    per_class_f1["readable_label"],
    per_class_f1["f1-score"],
    color="#3A8C5C",
    edgecolor="black",
    linewidth=0.65,
    zorder=3
)

ax.axvline(
    final_calibration_metrics["macro_f1"],
    color="#8B0000",
    linestyle="--",
    linewidth=1.5,
    label=(
        "Macro-F1 = "
        f"{final_calibration_metrics['macro_f1']:.3f}"
    )
)

ax.set_xlim(
    max(0.0, per_class_f1["f1-score"].min() - 0.08),
    1.01
)

ax.set_xlabel("F1-score")
ax.set_ylabel("Crop-disease class")

ax.set_title(
    "Per-class calibration F1-score for selected EfficientNetV2-S",
    pad=10
)

paper_axes(ax)

legend = ax.legend(
    loc="lower right",
    fancybox=False,
    borderpad=0.8
)

legend.get_frame().set_linewidth(0.9)

plt.tight_layout()

figure_path = FIG_DIR / "fig_05_proposed_calibration_per_class_f1.png"
pdf_path = FIG_DIR / "fig_05_proposed_calibration_per_class_f1.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)
print("Saved:", pdf_path)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))

ax.set_xlim(0, 13)
ax.set_ylim(0, 6)
ax.axis("off")

stages = [
    {
        "x": 0.45,
        "y": 2.15,
        "w": 1.6,
        "h": 1.7,
        "color": "#D9EAF7",
        "title": "Input image",
        "detail": "384 × 384 × 3"
    },
    {
        "x": 2.45,
        "y": 2.15,
        "w": 1.8,
        "h": 1.7,
        "color": "#CFE8D5",
        "title": "Stem",
        "detail": "Convolution + BN\nSiLU activation"
    },
    {
        "x": 4.70,
        "y": 1.65,
        "w": 2.1,
        "h": 2.7,
        "color": "#FBE5C8",
        "title": "EfficientNetV2-S",
        "detail": "Fused-MBConv and\nMBConv feature stages"
    },
    {
        "x": 7.25,
        "y": 2.15,
        "w": 1.8,
        "h": 1.7,
        "color": "#E8D9F1",
        "title": "Global pooling",
        "detail": "Feature embedding"
    },
    {
        "x": 9.50,
        "y": 2.15,
        "w": 1.6,
        "h": 1.7,
        "color": "#F6D6D6",
        "title": "Dropout",
        "detail": "p = 0.35"
    },
    {
        "x": 11.55,
        "y": 2.15,
        "w": 1.1,
        "h": 1.7,
        "color": "#D6E4F0",
        "title": "Classifier",
        "detail": "19 logits"
    }
]

for stage in stages:
    rectangle = plt.Rectangle(
        (stage["x"], stage["y"]),
        stage["w"],
        stage["h"],
        facecolor=stage["color"],
        edgecolor="black",
        linewidth=1.2,
        zorder=3
    )

    ax.add_patch(rectangle)

    ax.text(
        stage["x"] + stage["w"] / 2,
        stage["y"] + stage["h"] * 0.63,
        stage["title"],
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax.text(
        stage["x"] + stage["w"] / 2,
        stage["y"] + stage["h"] * 0.32,
        stage["detail"],
        ha="center",
        va="center",
        fontsize=9
    )

for left_stage, right_stage in zip(stages[:-1], stages[1:]):
    start_x = left_stage["x"] + left_stage["w"]
    start_y = left_stage["y"] + left_stage["h"] / 2

    end_x = right_stage["x"]
    end_y = right_stage["y"] + right_stage["h"] / 2

    ax.annotate(
        "",
        xy=(end_x - 0.05, end_y),
        xytext=(start_x + 0.05, start_y),
        arrowprops={
            "arrowstyle": "->",
            "lw": 1.6,
            "color": "black"
        }
    )

ax.text(
    6.5,
    5.35,
    "Proposed EfficientNetV2-S pipeline for Main19 crop-disease classification",
    ha="center",
    va="center",
    fontsize=14,
    fontweight="bold"
)

ax.text(
    6.5,
    0.65,
    "Checkpoint selection: calibration macro-F1 only; locked internal test not used during training",
    ha="center",
    va="center",
    fontsize=10,
    style="italic"
)

plt.tight_layout()

figure_path = FIG_DIR / "fig_06_proposed_architecture_diagram.png"
pdf_path = FIG_DIR / "fig_06_proposed_architecture_diagram.pdf"

plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.savefig(pdf_path, dpi=300, bbox_inches="tight")
plt.show()
plt.close()

print("Saved:", figure_path)
print("Saved:", pdf_path)

In [ ]:
run_metadata = {
    "notebook": "05_proposed_efficientnet_training",
    "completed_utc": datetime.now(timezone.utc).isoformat(),
    "model": "EfficientNetV2-S",
    "best_epoch": int(best_epoch),
    "best_calibration_macro_f1": float(best_macro_f1),
    "final_calibration_metrics": {
        key: float(value)
        for key, value in final_calibration_metrics.items()
    },
    "checkpoint_path": str(best_checkpoint_path),
    "training_history_path": str(history_path),
    "calibration_predictions_path": str(prediction_path),
    "internal_test_used": False,
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "torchvision_version": __import__("torchvision").__version__,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__
}

run_metadata_path = METADATA_DIR / "proposed_run_metadata.json"

with open(run_metadata_path, "w") as file:
    json.dump(run_metadata, file, indent=2)

print(json.dumps(run_metadata, indent=2))

In [ ]:
assert best_checkpoint_path.exists()
assert history_path.exists()
assert final_metrics_path.exists()
assert report_path.exists()
assert prediction_path.exists()
assert run_metadata_path.exists()

assert len(train_df) > 0
assert len(calibration_df) > 0

assert "internal_test" not in set(train_df["final_split"])
assert "internal_test" not in set(calibration_df["final_split"])

print("=" * 78)
print("NOTEBOOK 05 COMPLETED")
print("=" * 78)

print("\nSelected primary model")
print("Architecture: EfficientNetV2-S")
print("Best epoch:", best_epoch)
print(
    "Calibration macro-F1:",
    round(best_macro_f1, 6)
)

print("\nProtocol confirmation")
print("Training images:", len(train_df))
print("Calibration images:", len(calibration_df))
print("Internal-test records loaded: 0")

print("\nOutput directory:")
print(OUTPUT_DIR)

print("\nNext step:")
print(
    "Run a separate final-evaluation notebook only after this "
    "architecture is selected. It may load the locked internal-test "
    "manifest exactly once."
)